In [1]:
# Estrategia Lay Away

import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data_total/dados_betfair.csv", sep=";")

In [3]:
datatest = data[['League', 'Season', 'Home', 'Away', 'Goals_H_FT', 'Goals_A_FT', 'Odd_H_Back', 'Odd_A_Back', 'Odd_D_Back', 'Odd_A_Lay']].copy()

#datatest.to_csv("../test_estrat/data/datatest.csv", index=False, sep=";")

# Criação de variaveis para análise
datatest['VAR1'] = round(np.sqrt((datatest['Odd_H_Back'] - datatest['Odd_A_Back']) ** 2), 2)
datatest['VAR2'] = round(np.degrees(np.arctan((datatest['Odd_A_Back'] - datatest['Odd_H_Back']) / 2)), 2)
datatest['VAR3'] = round(np.degrees(np.arctan((datatest['Odd_D_Back'] - datatest['Odd_A_Back']) / 2)), 2)

# Verificadbo se o time da casa venceu ou não
datatest['Lay_Away'] = np.where((datatest['Goals_H_FT'] >= datatest['Goals_A_FT']), 1, 0)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_A_Lay'] - 1) if row['Lay_Away'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

# fILTRO
datatest = datatest[(datatest['VAR1'] >= 6.30) & (datatest['VAR2'] >= 71) & (datatest['VAR3'] >= -62) & (datatest['Odd_A_Lay'] > 8) & (datatest['Odd_A_Lay'] <= 17)]

# Quantidade de apostas
num_apostas = len(datatest)
print(f"Número de apostas: {num_apostas}")

# Quantidade de apostas vencedoras
num_vitorias = datatest['Lay_Away'].sum()
print(f"Número de apostas vencedoras: {num_vitorias}")

# Taxa de acerto
taxa_acerto = (num_vitorias / num_apostas) * 100
print(f"Taxa de acerto: {taxa_acerto:.2f}%")

# Somar o lucro total
lucro_total = datatest['Profit'].sum()
print(f"Lucro total: {lucro_total}")

# Calcular Yield
yield_value = (lucro_total / (num_apostas * STAKE)) * 100
#print(f"Yield: {yield_value:.2f}%")

# Medir o Drawdown
datatest['Cumulative_Profit'] = datatest['Profit'].cumsum()
datatest['Drawdown'] = datatest['Cumulative_Profit'] - datatest['Cumulative_Profit'].cummax()
max_drawdown = datatest['Drawdown'].min()
print(f"Drawdown máximo: {round(max_drawdown, 2)}%")

datatest.head()

Número de apostas: 135
Número de apostas vencedoras: 128
Taxa de acerto: 94.81%
Lucro total: 68.91999999999996
Drawdown máximo: -16.04%


,League,Season,Home,Away,Goals_H_FT,Goals_A_FT,Odd_H_Back,Odd_A_Back,Odd_D_Back,Odd_A_Lay,VAR1,VAR2,VAR3,Lay_Away,Profit,Cumulative_Profit,Drawdown
12,ITALY 1,2023/2024,AS Roma,Sassuolo,1,0,1.47,8.2,5.0,8.4,6.73,73.45,-57.99,1,0.94,0.94,0.0
42,GERMANY 1,2023/2024,RB Leipzig,Mainz,0,0,1.38,9.2,5.8,9.4,7.82,75.65,-59.53,1,0.94,1.88,0.0
82,ENGLAND 1,2023/2024,Liverpool,Brighton,2,1,1.38,8.4,6.2,8.6,7.02,74.10,-47.73,1,0.94,2.82,0.0
120,GERMANY 1,2023/2024,Mainz,Darmstadt,4,0,1.47,8.0,5.0,8.2,6.53,72.97,-56.31,1,0.94,3.76,0.0
172,GERMANY 1,2023/2024,RB Leipzig,Wolfsburg,3,0,1.38,9.2,5.8,9.4,7.82,75.65,-59.53,1,0.94,4.70,0.0
